# Main notebook with NER fine-tuning

In [1]:
# import sys

# !{sys.executable} -m pip install seqeval pyconll

## Imports

In [2]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import pyconll
from transformers import (
    pipeline,
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForTokenClassification)
from datasets import load_dataset, DatasetDict, Dataset
from seqeval.metrics import classification_report, accuracy_score
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

## Load data

In [3]:
main_data_path = Path("./conll_data_for_finetuning")

In [4]:
def load_custom_conll(file_path):
    dataset = []
    current_tokens = []
    current_tags = []
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            # Skip comments
            if line.startswith("#"):
                continue
                
            if not line:
                if current_tokens:
                    dataset.append({"tokens": current_tokens, "ner_tags": current_tags})
                    current_tokens = []
                    current_tags = []
            else:
                parts = line.split() 
                if len(parts) >= 2:
                    current_tokens.append(parts[0])  
                    current_tags.append(parts[-1])   
                    
        if current_tokens:
            dataset.append({"tokens": current_tokens, "ner_tags": current_tags})
            
    return dataset

# Create lists with train and eval data
train_data = load_custom_conll(main_data_path / "train.conll")
eval_data = load_custom_conll(main_data_path / "eval.conll")

# Create train and eval datasets
train_dataset = Dataset.from_list(train_data)
eval_dataset = Dataset.from_list(eval_data)

# Main dataset
dataset = DatasetDict({
    "train": train_dataset,
    "validation": eval_dataset
})

## Load tokenizer and model

In [5]:
model_name = "dslim/bert-base-NER"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Create dictionary for NER label: index

In [6]:
unique_ner_tags = set()

for i in range(len(train_dataset)):
    unique_ner_tags |= set(train_dataset[i]["ner_tags"])
    # break

print(f"Unique tags that are used in this NER task are: {unique_ner_tags}")

Unique tags that are used in this NER task are: {'O', 'I-MISC', 'B-LOC', 'B-MISC', 'I-ORG', 'B-ORG', 'B-PER', 'I-PER', 'I-LOC'}


In [7]:
label_list = ["O", "B-MISC", "I-MISC", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
label_to_id = {label: i for i, label in enumerate(label_list)}

## Tokenize and align labels

In [8]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        # padding="max_length",
        # max_length=512
    )
    
    labels = []
    for i, tags_list in enumerate(examples["ner_tags"]):
        # tags_list - the list of ner labels: ["O", "B-ORG", ...]
        # word_ids - the list of indices of words in the original text (None for CLS, SEP, PAD and etc)
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        
        previous_word_idx = None
        label_ids = []

        # go through indices
        for word_idx in word_ids:
            # for CLS, PAD and etc: use -100
            if word_idx is None:
                label_ids.append(-100)
            # If we encounter word
            elif word_idx != previous_word_idx:
                label_ids.append(label_to_id[tags_list[int(word_idx)]])
            # If we have already seen this word
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    # print(labels)
    tokenized_inputs["labels"] = labels
    
    return tokenized_inputs

Tokenize dataset and align NER labels.

In [9]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels, 
    batched=True,          
    batch_size=32           
)

Map:   0%|          | 0/832 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

## Training

In [10]:
# Function to compute metrics 
def compute_metrics(eval_preds):
    pred_logits, labels = eval_preds
    pred_logits = np.argmax(pred_logits, axis=2)
    
    predictions = []
    true_labels = []
    
    for prediction, label in zip(pred_logits, labels):
        pred_tags = []
        ref_tags = []
        for p, l in zip(prediction, label):
            if l != -100:
                pred_tags.append(label_list[p])
                ref_tags.append(label_list[l])
        if pred_tags:  
            predictions.append(pred_tags)
            true_labels.append(ref_tags)
    
    if not predictions:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "accuracy": 0.0,
        }
    
    results = classification_report(true_labels, predictions, output_dict=True)
    
    return {
        "precision": results["micro avg"]["precision"],
        "recall": results["micro avg"]["recall"],
        "f1": results["micro avg"]["f1-score"],
        "accuracy": accuracy_score(true_labels, predictions),
    }

In [11]:
training_args = TrainingArguments(
    output_dir="./ner_finetuned_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    weight_decay=0.01,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr": 5e-6},
    warmup_steps=40,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    save_total_limit=1,
    dataloader_pin_memory=False,
    use_cpu=True,
    seed=42,
)

Main training part

In [12]:
import warnings

warnings.filterwarnings("ignore")

In [13]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer) # , padding=True, max_length=512,)

# Train 
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.059500,0.813725,0.825146,0.819396,0.983020
2,No log,0.056072,0.813089,0.857310,0.834614,0.984208
3,No log,0.055532,0.815453,0.857895,0.836136,0.984272


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=156, training_loss=0.39456851665790266, metrics={'train_runtime': 902.0724, 'train_samples_per_second': 2.767, 'train_steps_per_second': 0.173, 'total_flos': 302475240783900.0, 'train_loss': 0.39456851665790266, 'epoch': 3.0})

In [14]:
model.save_pretrained("./ner_finetuned_model")
tokenizer.save_pretrained("./ner_finetuned_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./ner_finetuned_model/tokenizer_config.json',
 './ner_finetuned_model/tokenizer.json')

## Evaluate NER finetuned model

Load data for examples

In [15]:
example_df = pd.read_csv("./clean_data_for_section_tags_modelling/df_final_clean_filtered_new_new_v1.csv")

In [16]:
def create_main_text(row):
    title = str(row['title']).strip()
    body_text = str(row['body_text']).strip()
    
    return f"{title}  {body_text}"

main_text = example_df.apply(lambda row: create_main_text(row), axis=1)

Use fine-tuned NER model

In [17]:
ner_pipeline = pipeline(
    "ner",
    model="./ner_finetuned_model",
    aggregation_strategy="simple",
    device=0,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [18]:
for i in range(5):
    cur_text = main_text.iloc[i]
    ner_ans = ner_pipeline(cur_text)

    print(f"[TEXT]: {cur_text}\n\n")
    for dct in ner_ans:
        print(f"word: '{dct['word']}', entity_group: '{dct['entity_group']}'\n")
    print("\n")

[TEXT]: Tell us: what have you been reading this month?  As part of The Guardian’s “what we’re reading” series, we would like to hear about the books you’ve particularly enjoyed this month. Have you read a book in recent weeks – fiction or non-fiction – that you’d recommend? Tell us all about it below. Share your recommendations You can get in touch by filling in the form below. Your responses are secure as the form is encrypted and only the Guardian has access to your contributions. One of our journalists will be in contact before we publish, so please do leave contact details. If you’re having trouble using the form, click here. Read terms of service here and privacy policy here.


word: 'Guardian', entity_group: 'ORG'

word: 'Guardian', entity_group: 'ORG'



[TEXT]: Teenage boys in UK ‘stuck’ reading primary-level books while girls’ tastes expand  Teenage boys are “stuck” reading primary school books such as Diary of a Wimpy Kid, while girls their age are moving on to a wider range